In [119]:
#import
import pandas as pd
import rdata
import numpy as np
import tensorflow as tf

from pathlib import Path
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers



In [109]:

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

# URL base del repositorio oficial donde están los archivos .rda
BASE_URL = "https://github.com/paezha/idealista18/raw/master/data"

# Carpeta local donde guardaremos los archivos descargados
DATA_DIR = Path("idealista18_data")
DATA_DIR.mkdir(exist_ok=True)

# Relación entre ciudad y archivo de ventas correspondiente
SALE_FILES = {
    "Barcelona": "Barcelona_Sale.rda",
    "Madrid": "Madrid_Sale.rda",
    "Valencia": "Valencia_Sale.rda",
}


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def download_file(filename: str) -> Path:
    """
    Descarga un archivo desde el repositorio oficial si no existe en local.

    Parameters
    ----------
    filename : str
        Nombre del archivo a descargar, por ejemplo 'Madrid_Sale.rda'.

    Returns
    -------
    Path
        Ruta local al archivo descargado.
    """
    local_path = DATA_DIR / filename

    if not local_path.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Descargando {filename}...")
        urlretrieve(url, local_path)

    return local_path


def load_rda(filename: str) -> dict:
    """
    Carga un archivo .rda y devuelve su contenido como diccionario.

    Nota:
    Un archivo .rda puede contener uno o varios objetos de R.
    Esta librería normalmente los devuelve en un diccionario
    donde la clave es el nombre del objeto almacenado.

    Parameters
    ----------
    filename : str
        Nombre del archivo .rda.

    Returns
    -------
    dict
        Diccionario con los objetos cargados desde el archivo.
    """
    local_path = download_file(filename)
    data = rdata.read_rda(str(local_path), default_encoding="utf8")
    return data


def load_city_sales(city_name: str, filename: str) -> pd.DataFrame:
    """
    Carga el dataset de ventas de una ciudad y añade una columna CITY.

    Parameters
    ----------
    city_name : str
        Nombre de la ciudad, por ejemplo 'Madrid'.
    filename : str
        Nombre del archivo .rda asociado.

    Returns
    -------
    pd.DataFrame
        DataFrame con los datos de ventas de esa ciudad.
    """
    rda_content = load_rda(filename)

    # El objeto dentro del .rda suele llamarse igual que el archivo sin extensión
    object_name = filename.replace(".rda", "")

    if object_name not in rda_content:
        raise KeyError(
            f"No se encontró el objeto '{object_name}' dentro de '{filename}'. "
            f"Objetos disponibles: {list(rda_content.keys())}"
        )

    df = rda_content[object_name].copy()

    # Añadimos una columna para que el modelo sepa de qué ciudad viene cada vivienda
    df["CITY"] = city_name

    return df


def load_all_sales() -> pd.DataFrame:
    """
    Carga y concatena los datasets de ventas de Barcelona, Madrid y Valencia.

    Returns
    -------
    pd.DataFrame
        DataFrame unificado con las 3 ciudades.
    """
    city_dfs = []

    for city_name, filename in SALE_FILES.items():
        print(f"Cargando datos de {city_name}...")
        city_df = load_city_sales(city_name, filename)
        city_dfs.append(city_df)

    df_all = pd.concat(city_dfs, ignore_index=True)
    return df_all


# ============================================================
# CARGA PRINCIPAL
# ============================================================

# Cargar las tres ciudades en un único DataFrame
df = load_all_sales()


# ============================================================
# INSPECCIÓN BÁSICA
# ============================================================

print("\nPrimeras filas:")
print(df.head())

print("\nDimensiones del dataset unificado:")
print(df.shape)

print("\nCiudades incluidas:")
print(df["CITY"].value_counts())

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

Cargando datos de Barcelona...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi

Cargando datos de Madrid...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi

Cargando datos de Valencia...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi


Primeras filas:
                 ASSETID  PERIOD     PRICE    UNITPRICE  CONSTRUCTEDAREA  \
0  A11898131848556022319  201803  323000.0  3845.238095               84   
1  A18099432772155664747  201803  217000.0  2583.333333               84   
2   A2003099089407882787  201803  114000.0  1407.407407               81   
3   A1010373782315301134  201803  378000.0  4784.810127               79   
4  A12978912200216838006  201803  434000.0  3909.909910              111   

   ROOMNUMBER  BATHNUMBER  HASTERRACE  HASLIFT  HASAIRCONDITIONING  ...  \
0           4           1           1        1                   1  ...   
1           3           2           0        1                   1  ...   
2           2           1           0        1                   1  ...   
3           2           1           0        1                   0  ...   
4           4           2           1        1                   1  ...   

   BUILTTYPEID_3  DISTANCE_TO_CITY_CENTER  DISTANCE_TO_METRO  \
0          

In [110]:
df["CITY"].value_counts()


CITY
Madrid       94815
Barcelona    61486
Valencia     33622
Name: count, dtype: int64

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189923 entries, 0 to 189922
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   ASSETID                        189923 non-null  object 
 1   PERIOD                         189923 non-null  Int32  
 2   PRICE                          189923 non-null  float64
 3   UNITPRICE                      189923 non-null  float64
 4   CONSTRUCTEDAREA                189923 non-null  Int32  
 5   ROOMNUMBER                     189923 non-null  Int32  
 6   BATHNUMBER                     189923 non-null  Int32  
 7   HASTERRACE                     189923 non-null  Int32  
 8   HASLIFT                        189923 non-null  Int32  
 9   HASAIRCONDITIONING             189923 non-null  Int32  
 10  AMENITYID                      189923 non-null  Int32  
 11  HASPARKINGSPACE                189923 non-null  Int32  
 12  ISPARKINGSPACEINCLUDEDINPRICE 

In [122]:

# DATOS

target = "PRICE"

features = [
    "CONSTRUCTEDAREA",
    "ROOMNUMBER",
    "BATHNUMBER",
    "HASTERRACE",
    "HASLIFT",
    "HASAIRCONDITIONING",
    "HASPARKINGSPACE",
    "HASBOXROOM",
    "HASWARDROBE",
    "HASSWIMMINGPOOL",
    "HASGARDEN",
    "ISDUPLEX",
    "ISSTUDIO",
    "ISINTOPFLOOR",
    "LATITUDE",
    "LONGITUDE",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "CITY",
]

df_model = df[features + [target]].copy()
df_model = df_model.dropna()

# Convertir CITY a variables dummy
df_model = pd.get_dummies(df_model, columns=["CITY"], drop_first=True)
df_model.columns = [str(col) for col in df_model.columns]

# Separar X e y
X = df_model.drop(columns=[target]).copy()
y = df_model[target].copy()

# Train / temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Validation / test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

#Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Train: (132946, 20) (132946,)
Val: (28488, 20) (28488,)
Test: (28489, 20) (28489,)


In [ ]:

def crear_modelo(dropout_rate, input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 13,057 (51.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 50638159872.0000 - mae: 126316.0000 - val_loss: 23660224512.0000 - val_mae: 87286.1406
Epoch 2/3
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 22405957632.0000 - mae: 84837.6094 - val_loss: 21777410048.0000 - val_mae: 81864.8594
Epoch 3/3
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 4s 954us/step - loss: 21514913792.0000 - mae: 82476.5469 - val_loss: 20815497216.0000 - val_mae: 80657.8672
